# **(1) Ingession pipeline**

In [1]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma

C:\Users\Lapmart\AppData\Local\Temp\ipykernel_10508\694688087.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader


### **Configurations**

In [3]:
doc_path = "docs"
db_path = "db/chroma_db"
embedding_model_path = "../../../Data/HuggingFaceEmbeddings Models/all-mpnet-base-v2"

chunk_size = 800
chunk_overlap = 0

In [6]:
# import os 
# os.listdir(embedding_model_path)

### **Load documents**

In [7]:
print(f"Loading docs from '{doc_path}'")

loader = DirectoryLoader(
    path=doc_path,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)

documents = loader.load()

if len(documents)==0:
    raise FileNotFoundError(f"No any .txt files in path '{doc_path}' directory")

Loading docs from 'docs'


In [8]:
for i, doc in enumerate(documents[:2]):
    print("\n###", type(doc))
    print(f"Doc {i+1}")
    print(f"Source : {doc.metadata['source']}")
    print(f"Content length : {len(doc.page_content)} characters\n")


### <class 'langchain_core.documents.base.Document'>
Doc 1
Source : docs\about us.txt
Content length : 14828 characters


### <class 'langchain_core.documents.base.Document'>
Doc 2
Source : docs\Annual Report.txt
Content length : 33942 characters



### **Split documents into chunks**

In [9]:
text_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

chunks = text_splitter.split_documents(documents=documents) # List of Document()

Created a chunk of size 1155, which is longer than the specified 800
Created a chunk of size 890, which is longer than the specified 800
Created a chunk of size 1409, which is longer than the specified 800
Created a chunk of size 870, which is longer than the specified 800
Created a chunk of size 815, which is longer than the specified 800
Created a chunk of size 1094, which is longer than the specified 800
Created a chunk of size 1134, which is longer than the specified 800
Created a chunk of size 803, which is longer than the specified 800
Created a chunk of size 1033, which is longer than the specified 800


In [10]:
for i, chunk in enumerate(chunks[0:5], 1):
    print(f"DOC : {i}")
    print(f"metadatac : {chunk.metadata}")
    print(f"Chunk length : {len(chunk.page_content)} chars")
    print(f"Chunk content : {chunk.page_content}")
    print("-" * 50)

    if len(chunks) > 5:
        print(f"more {len(chunks) -5} are there more...")

DOC : 1
metadatac : {'source': 'docs\\about us.txt'}
Chunk length : 722 chars
Chunk content : ================================================================================
COMMERCIAL BANK OF CEYLON PLC - COMPLETE WEBSITE CONTENT (FOR RAG)

SECTION 1: LANGUAGE & MAIN NAVIGATION

Languages: සිංහල | தமிழ்

Main Navigation:
- About us
- Branches/ATMs
- Investors
- News
- Careers
- Contact us
- Overseas Operations
- Search

Accessibility Options

Banking Sections:
- Personal Banking
- Business Banking
- Non Resident Banking
- Services
- ComBank Digital
- Online Banking
--------------------------------------------------
more 81 are there more...
DOC : 2
metadatac : {'source': 'docs\\about us.txt'}
Chunk length : 759 chars
Chunk content : Contact Tools:
- Contact (Page)
- Mail (Email link)
- Branches/ATMs (Locator)
- Calculators
- Exchange Rates

SECTION 2: ABOUT US - OVERVIEW & PHILOSOPHY

About Us: Over a Century of Banking.
Having set a benchmark in banking in Sri Lanka we have set stan

### **Loading embedding model**

In [11]:
embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_path,
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("### Embedding model loaded")

C:\Users\Lapmart\AppData\Local\Temp\ipykernel_10508\3386060054.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Embedding model loaded


### **Creating Vector DB**

In [15]:
# Delete old DB

import shutil

try :
    shutil.rmtree(db_path)  # delete old vector DB
except FileNotFoundError or NameError:
    pass

In [16]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=db_path,
    collection_metadata={"hnsw:space" : "cosine"}
)

print("### vector db created")

### vector db created
